# Aprendizado de Máquina — Lista prática E1

## Análise de Agrupamento: $k$-Médias

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Pela primeira vez no curso **não há $y$**. Sem resposta não há risco, não há
validação cruzada e não há resposta certa — o que muda completamente o que
significa "avaliar" um método.

Você vai escrever o algoritmo de Lloyd em dez linhas, conferir contra o
`scikit-learn`, e descobrir que a sua implementação está certa e o resultado
dela é ruim:

> **o $k$-médias converge sempre, e converge para um mínimo local. A diferença
> entre a melhor e a pior execução, nestes dados, é de um fator 12.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image, make_blobs
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — Lloyd em dez linhas

O algoritmo alterna dois passos até nada mudar:

1. **atribuir** cada ponto ao centroide mais próximo;
2. **atualizar** cada centroide para a média dos pontos atribuídos a ele.

A Lista Teórica E1 provou que nenhum dos dois aumenta a inércia
$J = \sum_i \lVert x_i - \mu_{c(i)}\rVert^2$.

In [ ]:
X, grupos_verdadeiros = make_blobs(n_samples=600, centers=4,
                                   cluster_std=1.0, random_state=2026)
print("X:", X.shape, " grupos verdadeiros:", len(set(grupos_verdadeiros)))

In [ ]:
def lloyd(X, K, mu, max_iter=100):
    historico = []
    for _ in range(max_iter):
        # passo 1: distancia de cada ponto (linha) a cada centroide (coluna)
        D = np.linalg.norm(X[:, None, :] - mu[None, :, :], axis=2)
        atribuicao = D.argmin(axis=1)                            # (a)

        # passo 2: cada centroide vira a media do seu grupo
        novo = np.array([X[atribuicao == k].mean(axis=0)         # (b)
                         if (atribuicao == k).any() else mu[k]
                         for k in range(K)])

        historico.append(float(np.sum((X - novo[atribuicao]) ** 2)))
        if np.allclose(novo, mu):                                # (c) criterio de parada
            break
        mu = novo
    return novo, atribuicao, historico


rng = np.random.default_rng(2026)
mu_inicial = X[rng.choice(len(X), 4, replace=False)]               # 4 pontos ao acaso

mu, atribuicao, historico = lloyd(X, 4, mu_inicial.copy())
print(f"convergiu em {len(historico)} iteracoes")
print(f"inercia final: {historico[-1]:.4f}")
print("historico:", [round(h, 2) for h in historico])

Confira contra o `scikit-learn`, dando a ele **exatamente a mesma
inicialização** e proibindo-o de reiniciar.

In [ ]:
km = KMeans(n_clusters=4, init=mu_inicial, n_init=1, random_state=0).fit(X)   # (a) e (b)

print(f"sklearn: {km.inertia_:.4f}")
print(f"a mao:   {historico[-1]:.4f}")
print(f"diferenca: {abs(km.inertia_ - historico[-1]):.2e}")

Deve imprimir `convergiu em 7 iteracoes`, `inercia final: 4013.0571`, o histórico
`[8483.08, 7703.86, 6143.28, 4178.3, 4013.46, 4013.06, 4013.06]`, e depois a
comparação com diferença da ordem de $10^{-13}$.

Duas coisas.

**A implementação está certa**: o `KMeans` faz literalmente o laço que você
escreveu, e com a mesma inicialização chega ao mesmo número até a precisão de
máquina.

**O histórico é monótono**, como a Lista Teórica E1 provou: $8483 \to 7704 \to
6143 \to 4178 \to 4013 \to 4013$. Ele nunca sobe, e para quando para de descer.

Guarde o $4013{,}06$. O Exercício 2 vai mostrar que ele é ruim.

---
## Exercício 2 — inércia e silhueta contra $K$

Agora deixe o `scikit-learn` escolher a inicialização (padrão: `k-means++`, com
`n_init=20` reinícios) e varra $K$ de 2 a 8.

In [ ]:
print("   K    inercia    silhueta")
inercias, silhuetas = [], []

for K in range(2, 9):
    modelo = KMeans(n_clusters=K, n_init=20, random_state=2026).fit(X)          # (a)
    inercias.append(modelo.inertia_)
    silhuetas.append(silhouette_score(X, modelo.labels_))                       # (b)
    print(f"  {K:2d}  {inercias[-1]:9.2f}    {silhuetas[-1]:.4f}")

print(f"\nK que maximiza a silhueta: {range(2, 9)[int(np.argmax(silhuetas))]}")  # (c)

Deve imprimir:

```
   K    inercia    silhueta
   2   16254.88    0.6998
   3    4147.94    0.7478
   4    1288.11    0.7764
   5    1152.83    0.6463
   6    1032.11    0.5386
   7     923.95    0.4531
   8     822.96    0.3288

K que maximiza a silhueta: 4
```

**A inércia cai sem parar** — $16\,255 \to 823$ — exatamente como o Exercício 3(b)
da Lista Teórica E1 provou que teria de cair. Ela não tem máximo interior e
portanto não serve como critério: se você a minimizasse, escolheria $K=600$.

**A silhueta tem um máximo claro em $K=4$** ($0{,}7764$), que é o número
verdadeiro de nuvens do `make_blobs`. Ela acerta sem que ninguém lhe conte.

E agora compare o $1288{,}11$ desta tabela com o $4013{,}06$ do Exercício 1: **é
o mesmo $K=4$, nos mesmos dados**, e a sua implementação à mão terminou com
inércia **três vezes maior**. A implementação não estava errada — a
inicialização é que era ruim, e o algoritmo convergiu, obedientemente, para um
mínimo local.

> **Sua vez.** Desenhe as duas curvas (inércia e silhueta) contra $K$, lado a
> lado. Dá para ver um "cotovelo" na primeira? Ele cai em $K=4$?

---
## Exercício 3 — quanto a inicialização custa

Meça diretamente a variabilidade: rode 30 vezes com **um** reinício e
inicialização aleatória, e compare com o padrão do `scikit-learn`.

In [ ]:
inercias_1 = np.array([
    KMeans(n_clusters=4, init="random", n_init=1, random_state=s).fit(X).inertia_   # (a)
    for s in range(30)
])

melhor = KMeans(n_clusters=4, n_init=50, random_state=0).fit(X).inertia_

print(f"30 execucoes com n_init=1 e init aleatoria:")
print(f"  melhor: {inercias_1.min():.2f}")
print(f"  pior:   {inercias_1.max():.2f}   ({inercias_1.max() / inercias_1.min():.1f}x a melhor)")
print(f"  quantas ficaram >1% acima do melhor: "
      f"{int((inercias_1 > melhor * 1.01).sum())}/30")                              # (b)

print(f"\ncom n_init=10 (o padrao) e k-means++: "
      f"{KMeans(n_clusters=4, random_state=0).fit(X).inertia_:.2f}")

Deve imprimir:

```
30 execucoes com n_init=1 e init aleatoria:
  melhor: 1288.11
  pior:   16073.06   (12.5x a melhor)
  quantas ficaram >1% acima do melhor: 11/30
com n_init=10 (o padrao) e k-means++: 1288.11
```

**Onze das trinta execuções terminaram num mínimo local ruim**, e a pior delas
com inércia $12{,}5$ vezes a melhor — pior até que a solução com $K=2$ da tabela
do Exercício 2. Uma em cada três tentativas, com um único reinício, produz um
agrupamento que não serve.

Os dois remédios do `scikit-learn`, ambos ligados por padrão:

- **`n_init=10`**: roda o algoritmo dez vezes e devolve a de menor inércia. Como
  a chance de uma execução ser ruim é ~1/3, a chance de as dez serem ruins é
  $(1/3)^{10} \approx 2\times 10^{-5}$;
- **`init="k-means++"`**: em vez de sortear os centroides uniformemente, sorteia
  o primeiro ao acaso e cada seguinte com probabilidade proporcional ao quadrado
  da distância ao centroide mais próximo já escolhido — o que os espalha e evita
  o modo de falha mais comum, dois centroides caindo na mesma nuvem.

Aqui os dois juntos acertam o ótimo com $n_{\text{init}}=10$.

E vale a comparação de fundo com o resto do curso: em aprendizado supervisionado,
rodar o mesmo método duas vezes nos mesmos dados dá o mesmo resultado, e a
variabilidade vem da **amostra**. Aqui ela vem do **algoritmo**, e é preciso
controlá-la explicitamente.

---
## Exercício 4 — comprimindo uma imagem

Uma aplicação em que os "grupos" têm significado direto. Uma fotografia é uma
lista de pixels, cada um um ponto em $\mathbb{R}^3$ (vermelho, verde, azul).
Agrupar os pixels em $K$ grupos e substituir cada um pela cor do seu centroide é
**quantização de cor**: a imagem passa a usar uma paleta de $K$ cores.

In [ ]:
imagem = load_sample_image("china.jpg")
pixels = imagem.reshape(-1, 3) / 255.0                           # (a) uma linha por pixel, RGB

print(f"imagem: {imagem.shape}")
print(f"pixels: {pixels.shape[0]}")
print(f"cores distintas: {len(np.unique(pixels, axis=0))}")

In [ ]:
# ajustar em 4000 pixels sorteados basta, e e MUITO mais rapido que nos 273 mil
amostra = pixels[np.random.default_rng(0).choice(len(pixels), 4000, replace=False)]

bits_originais = imagem.size * 8
print("    K       EQM      tamanho       compressao")
for K in (4, 16, 64):
    modelo = KMeans(n_clusters=K, n_init=4, random_state=2026).fit(amostra)
    rotulos = modelo.predict(pixels)                             # (a) aplique a TODOS os pixels
    reconstruida = modelo.cluster_centers_[rotulos]              # (b) troque cada pixel pelo centroide

    eqm = np.mean((pixels - reconstruida) ** 2)
    bits = len(pixels) * np.ceil(np.log2(K)) + K * 3 * 8           # indices + paleta
    print(f"  {K:3d}   {eqm:.6f}    {100 * bits / bits_originais:5.1f}%      "
          f"{bits_originais / bits:.1f}x")

Deve imprimir `imagem: (427, 640, 3)`, `pixels: 273280`, `cores distintas: 96615`
e:

```
    K       EQM      tamanho       compressao
    4   0.007032      8.3%      12.0x
   16   0.001782     16.7%       6.0x
   64   0.000610     25.0%       4.0x
```

De **96 615 cores distintas** para 4, e a imagem ainda é reconhecível — com um
doze avos do espaço.

O balanço aqui é o de sempre, com outro nome. $K$ é o parâmetro de
complexidade: $K$ pequeno dá muita compressão e muita distorção (o EQM cai por um
fator de 11,5 ao ir de 4 para 64 cores), $K$ grande faz o contrário. A diferença
em relação ao resto do curso é que **não existe $K$ ótimo**: a escolha depende do
quanto de distorção a aplicação tolera, e não de um risco a minimizar.

Repare também no truque da célula: o `KMeans` foi ajustado em **4 000** pixels
sorteados e depois aplicado aos 273 280. Os centroides de uma amostra aleatória
de 4 000 pontos já estimam muito bem os centroides da população de pixels — é o
mesmo argumento de tamanho de amostra do resto do curso, aplicado a um problema
sem $y$.

> **Sua vez.** Mostre as quatro imagens lado a lado (original e $K=4,16,64$) com
> `ax.imshow(reconstruida.reshape(imagem.shape))`. A partir de qual $K$ você
> deixaria de notar a diferença?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o Lloyd à mão bate o `KMeans` até $10^{-13}$, e a inércia cai monotonicamente |
| 2 | a inércia cai sempre (16 255 → 823); a silhueta tem máximo em $K=4$, o valor verdadeiro |
| 2 | a sua execução do Exercício 1 deu 4013, contra 1288 do ótimo — **3× pior**, e correta |
| 3 | com `n_init=1`, **11 de 30** execuções ficam acima do ótimo; a pior, 12,5× |
| 4 | 96 615 cores viram 4, com $12\times$ de compressão |

**A seguir.** A Aula E2 ataca o outro problema sem $y$: em vez de agrupar
observações, reduzir o número de colunas — e vai reencontrar a mesma soma de
quadrados, decomposta de outro jeito.